# Calculating Effective Sample Sizes

**This notebook calculates overlap-weighted and stabilized IPTW ESS for each treatment arm**

In [1]:
import numpy as np
import pandas as pd

from iptw_survival import OverlapWeightSurvivalEstimator, IPTWSurvivalEstimator

## Import data 

### Import treatment assignment

In [2]:
treatment_df = pd.read_csv('../outputs/pembro_chemo_index.csv')

In [3]:
treatment_df.shape

(26405, 3)

In [4]:
treatment_df.head(3)

,PatientID,LineName,StartDate
0,FFFB7D51F655C,pembro,2023-10-04
1,FB62DC3D3C20F,pembro,2025-06-05
2,F15B1F9F90957,pembro,2018-07-31


### Import features

In [5]:
dtype_map = pd.read_csv('../outputs/pembro_chemo_features_dtypes.csv', index_col = 0).iloc[:, 0].to_dict()
features_df = pd.read_csv('../outputs/pembro_chemo_features_df.csv', dtype = dtype_map)

In [6]:
features_df.shape

(1208, 170)

### Merge dataframes

In [7]:
df = pd.merge(features_df, treatment_df, on = 'PatientID', how = 'left')

In [8]:
df.shape

(1208, 172)

In [9]:
df['StartDate'] = df['StartDate'].astype(str).str.split(' ').str[0]
df['StartDate'] = pd.to_datetime(df['StartDate'])

In [10]:
df['treatment_year'] = df['StartDate'].dt.year

In [11]:
df = df.query('treatment_year <= 2023')

In [12]:
df.shape

(1043, 173)

## Calculating overlap weights

In [13]:
df['treatment'] = np.where(df['LineName'] == 'pembro', 1, 0)

In [14]:
df.treatment.value_counts()

treatment
0    674
1    369
Name: count, dtype: int64

In [15]:
overlap_estimator = OverlapWeightSurvivalEstimator(normalize = None)

In [16]:
overlap_estimator.fit(df = df,
                      treatment_col = 'treatment',
                      cat_var = ['CrcSite'],
                      cont_var = ['days_diagnosis_to_met', 'GroupStage_mod', 'ecog_index', 'ses_mod', 'age', 'percent_change_weight', 'hemoglobin', 'albumin', 'alp', 'cea', 'van_walraven_score'],
                      binary_var = ['sex_male', 'ecog_index_na', 'ses_mod_na', 'medicaid', 'days_to_treatment_before_30d'],
                      lr_kwargs = {
                          'class_weight': 'balanced',
                          'solver': 'lbfgs',
                          'penalty': 'l2',
                          'max_iter': 1000,
                          'random_state': 42
                      },
                      clip_bounds = (0.01, 0.99),
                      use_missing_flags = False)

In [17]:
overlap_df = overlap_estimator.transform()

In [18]:
overlap_df.shape

(1043, 176)

## Calculating IPTW weights 

In [19]:
iptw_estimator = IPTWSurvivalEstimator()

In [20]:
iptw_estimator.fit(df = df,
                   treatment_col = 'treatment',
                   cat_var = ['CrcSite'],
                   cont_var = ['days_diagnosis_to_met', 'GroupStage_mod', 'ecog_index', 'ses_mod', 'age', 'percent_change_weight', 'hemoglobin', 'albumin', 'alp', 'cea', 'van_walraven_score'],
                   binary_var = ['sex_male', 'ecog_index_na', 'ses_mod_na', 'medicaid', 'days_to_treatment_before_30d'],
                   lr_kwargs = {
                       'class_weight': 'balanced',
                       'solver': 'lbfgs',
                       'penalty': 'l2',
                       'max_iter': 1000,
                       'random_state': 42
                   },
                   clip_bounds = (0.01, 0.99),
                   stabilized = True,
                   use_missing_flags = False)

In [21]:
iptw_df = iptw_estimator.transform()

In [22]:
iptw_df = iptw_df[['PatientID', 'iptw']]

In [23]:
iptw_df.shape

(1043, 2)

## Generating ESS table 

In [24]:
comb_weight_df = pd.merge(overlap_df, iptw_df, on = 'PatientID', how = 'left')

In [25]:
comb_weight_df.shape

(1043, 177)

In [26]:
def effective_sample_size(weights):
    weights = np.asarray(weights, dtype=float)
    weights = weights[np.isfinite(weights)]
    return weights.sum()**2 / np.sum(weights**2)

ess_table = (
    comb_weight_df
    .groupby("treatment")
    .agg(
        raw_n=("treatment", "size"),
        overlap_ess=("overlap_weight", effective_sample_size),
        iptw_ess=("iptw", effective_sample_size),
    )
    .reset_index()
)

ess_table[["overlap_ess", "iptw_ess"]] = (
    ess_table[["overlap_ess", "iptw_ess"]].round(3)
)

print(ess_table)

   treatment  raw_n  overlap_ess  iptw_ess
0          0    674      627.965   629.438
1          1    369      340.627   326.164
